# F6 · Combining Tables

*Foundations, notebook 6 of 6 · about 20 to 30 minutes*

**In this notebook you will learn to:**

- Combine two tables that share a column
- Tell an inner join from a left join
- Deal with the missing values a join creates
- Check a join for accidentally duplicated rows

**Before this notebook:** [F5: New Columns and Data Files](https://colab.research.google.com/github/icohr-lab/tutorials/blob/main/notebooks/F5_New_Columns_and_Data_Files.ipynb).

> **Before you start:** click **File > Save a copy in Drive** and work in your copy. Run a code cell by clicking the play button to its left, or press **Shift + Enter**. Go in order from top to bottom.

## 1. Why combine tables?

Health record data does not come as one big table. Demographics are in one table, diagnoses in another, prescriptions in a third. To study whether a drug is related to a disease, you need to bring them together. Combining tables is called a **join**.

Run the setup cell. (The lines printed by `library(tidyverse)` are normal.)

In [ ]:
library(tidyverse)

patients <- data.frame(
  Name = c("Josh", "Evelyn", "Olivia", "Liam", "Adrian", "Tamara", "Teddy", "Allison"),
  Age  = c(23, 18, 32, 56, 15, 25, 24, 16),
  City = c("Moscow", "Moscow", "Troy", "Viola", "Moscow", "Pullman", "Lewiston", "Moscow")
)
patients

Here is a second table. It has BMI measurements, but only for four of the eight patients.

In [ ]:
bmi <- data.frame(
  Name = c("Evelyn", "Liam", "Olivia", "Tamara"),
  BMI  = c(20.2, 27.9, 32.1, 21.6)
)
bmi

Both tables have a `Name` column. That shared column is what lets R match the rows: Evelyn's row in one table belongs with Evelyn's row in the other. The shared column is called the **key**.

## 2. Inner join: only the matches

`inner_join()` keeps only the people who appear in **both** tables. `by = "Name"` says which column to match on. Here, unlike in `select()` and `filter()`, the column name goes in quotation marks.

In [ ]:
inner_join(patients, bmi, by = "Name")

Four rows: the four people who have a BMI. Josh, Adrian, Teddy, and Allison have disappeared. An inner join can silently drop people, so use it only when that is what you want.

## 3. Left join: keep everyone in the first table

`left_join()` keeps **every row of the first (left) table** and adds information from the second table where there is a match.

**Predict first:** how many rows will the result have? What will be in the BMI column for Josh?

In [ ]:
left_join(patients, bmi, by = "Name")

All eight patients are still there. Where there was no BMI to add, R put `NA`: missing.

In research you usually start from your list of participants and add information to it, so `left_join()` is the join you will use most.

With the pipe, the same join is written like this. Read it as: take `patients`, and then left-join `bmi` onto it.

In [ ]:
patients_bmi <- patients %>% left_join(bmi, by = "Name")
patients_bmi

## 4. Joins create missing values

The join gave you four `NA` values, and from notebook F2 you know what they do to a calculation:

In [ ]:
mean(patients_bmi$BMI)

In [ ]:
mean(patients_bmi$BMI, na.rm = TRUE)

Before analyzing a joined table, find out how much is missing. `is.na()` works on a whole table as well as on a vector, and `colSums()` adds up each column. Together they count the missing values in every column at once.

In [ ]:
colSums(is.na(patients_bmi))

To look at the rows where a value is missing, put `is.na()` inside `filter()`:

In [ ]:
patients_bmi %>% filter(is.na(BMI))

To remove those rows, use `drop_na()`. Do this only when you have decided those people should not be in your analysis.

In [ ]:
patients_bmi %>% drop_na(BMI)

**Your turn 1.** Using `patients_bmi`, add a column called `Overweight` that says `"Yes"` if BMI is greater than 25 and `"No"` otherwise. Look at the result. What happened to the people with no BMI?

*Expected result: Olivia and Liam are Yes; Evelyn and Tamara are No; the other four are NA*

In [ ]:
# Your code here

## 5. The trap: a join can multiply rows

So far, each person appeared once in each table. Now look at a table of clinic visits, where a person can have several rows.

In [ ]:
visits <- data.frame(
  Name       = c("Josh", "Josh", "Evelyn", "Liam", "Liam", "Liam"),
  Visit_year = c(2019, 2021, 2020, 2018, 2019, 2022)
)
visits

**Predict first:** `patients` has 8 rows. How many rows will this join produce?

In [ ]:
patients_visits <- patients %>% left_join(visits, by = "Name")
patients_visits

In [ ]:
nrow(patients_visits)

Eleven rows, from a table of eight patients. Josh now appears twice and Liam three times, once for each visit. (The five patients with no visits have `NA`, as before.) Nothing is wrong with the join; it did what it was asked. But if you now counted rows to find out how many patients you have, or calculated the mean age, Josh and Liam would be counted more than once.

**This is the most common silent mistake when working with health record data.** No error message appears. The protection is a habit:

> **After every join, compare the number of rows with the number of different people.**

In [ ]:
nrow(patients_visits)                  # rows
n_distinct(patients_visits$Name)       # different people

`n_distinct()` is a shorter way of writing `length(unique(...))`. When the two numbers differ, some people appear more than once.

## 6. The same thing with a larger dataset

Now load two of the simulated research tables: the 100 people from the last notebook, and their diagnosis records.

In [ ]:
person_df <- read_csv("https://raw.githubusercontent.com/icohr-lab/aou-workshop/refs/heads/main/data/demographic_data.csv")
condition_df <- read_csv("https://raw.githubusercontent.com/icohr-lab/aou-workshop/refs/heads/main/data/conditions.csv")

In [ ]:
head(condition_df)

Each row of `condition_df` is one diagnosis record: who (`person_id`), what (`condition_source_value`), and when (`condition_start_date`). Because the data are simulated, some dates are in the future. Here, `person_id` is the key that links the two tables.

Do some people have more than one record?

In [ ]:
nrow(condition_df)                     # diagnosis records
n_distinct(condition_df$person_id)     # different people with a diagnosis

96 records, but only 30 different people. So joining this table to `person_df` will multiply rows:

In [ ]:
joined <- person_df %>% left_join(condition_df, by = "person_id")
nrow(joined)
n_distinct(joined$person_id)

166 rows for 100 people. To get a table with one row per person, you first have to reduce `condition_df` to one row per person, for example by keeping each person's earliest diagnosis. You will do that early in the next part of the course.

**Your turn 2.** Read the prescription records into a dataframe called `drug_df` from this address:

`https://raw.githubusercontent.com/icohr-lab/aou-workshop/refs/heads/main/data/drug_exposures.csv`

Find how many records it has and how many different people. Then start from `person_df` and left-join `drug_df` onto it, as was done with `condition_df` above. How many rows does the result have?

*Expected result: 267 records for 35 different people; the join has 332 rows*

In [ ]:
# Your code here

---
## Recap

| Code | What it does |
|---|---|
| `inner_join(a, b, by = "key")` | Keep only rows that match in both tables |
| `left_join(a, b, by = "key")` | Keep every row of `a`; add columns from `b` where they match |
| `a %>% left_join(b, by = "key")` | The same left join, written with the pipe |
| `colSums(is.na(df))` | Count missing values in every column |
| `filter(is.na(BMI))` | Show the rows where BMI is missing |
| `drop_na(BMI)` | Remove the rows where BMI is missing |
| `n_distinct(df$person_id)` | Number of different people |

---
## Solutions

Try each exercise before looking. There is often more than one correct way to write it.

**Your turn 1**

R cannot say whether an unknown BMI is greater than 25, so the answer is also unknown: `NA`. Missing values carry through into anything calculated from them.

```r
patients_bmi %>% mutate(Overweight = if_else(BMI > 25, "Yes", "No"))
```

**Your turn 2**

```r
drug_df <- read_csv("https://raw.githubusercontent.com/icohr-lab/aou-workshop/refs/heads/main/data/drug_exposures.csv")
nrow(drug_df)
n_distinct(drug_df$person_id)

joined_drug <- person_df %>% left_join(drug_df, by = "person_id")
nrow(joined_drug)
```

---
## Next

You have finished the Foundations. From here the course continues on one of two paths. Your instructor will tell you which one to follow.

- **Path A: Write It Yourself.** You write the code. Start with [A1: Cleaning Real Data](https://colab.research.google.com/github/icohr-lab/tutorials/blob/main/notebooks/A1_Cleaning_Real_Data.ipynb).
- **Path B: AI-Assisted.** You describe what you want to an AI assistant, then read and check the code it writes. Start with [B1: Your First Prompts](https://colab.research.google.com/github/icohr-lab/tutorials/blob/main/notebooks/B1_First_Prompts.ipynb).